# Recognize diagrams

Show each diagram independently to a model and ask **what data structure it communicates**, with confidence and visible evidence. Filenames and intended answers are hidden from the model. This collects recognition judgments; it does not calculate accuracy automatically.

Select the **spytial** Python environment as the notebook kernel. Run the cells from top to bottom. Start with **preview** to check the inputs without calling a model; switch to **evaluate** when ready. Every run is saved separately.

## 1. Load the notebook tools

These use the existing evaluation scripts, so the notebook and command-line versions behave consistently.

In [ ]:
from pathlib import Path
import csv
from datetime import datetime, timezone
import html
import json
import subprocess
import sys
from IPython.display import HTML, display

# Works when opened from the repository root or this notebook's folder.
EVAL_DIR = next((p for p in (Path.cwd() / "static-images/eval", Path.cwd())
                 if (p / "run_rq1.py").is_file()), None)
if EVAL_DIR is None:
    raise FileNotFoundError("Open this notebook from the repository root or static-images/eval.")

def show_table(rows, columns):
    def cell(value):
        if isinstance(value, (list, dict)):
            value = json.dumps(value, ensure_ascii=False)
        return html.escape(str(value if value is not None else "—"))
    header = "".join(f"<th>{cell(label)}</th>" for key, label in columns)
    body = "".join("<tr>" + "".join(f"<td>{cell(row.get(key))}</td>" for key, _ in columns) + "</tr>" for row in rows)
    display(HTML(f'<div style="overflow:auto;max-height:600px"><table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>'))


## 2. Choose settings

`preview` prepares the run without model calls. `evaluate` sends images to the selected model. For a first live check, set `LIMIT = 1`; use `None` for the full dataset.

Claude Code uses your existing login. Claude API uses `ANTHROPIC_API_KEY` from the environment and API billing. Keep credentials out of this notebook.

In [ ]:
MODE = "preview"                 # "preview" or "evaluate"
BACKEND = "claude-code"          # claude-code, claude-api, codex, or ollama
MODEL = None                    # None uses the configured model; otherwise enter a model name
LIMIT = None                    # Images; 1 for a quick live check, None for all
IMAGE_DIRECTORY = EVAL_DIR.parent
OUTPUT_ROOT = EVAL_DIR / "results"
CONFIG_FILE = EVAL_DIR / "config.json"


## 3. Inspect the prompt and run settings

The exact prompt and effective settings are saved with each run. Edit the versioned prompt file to change the rubric.

In [ ]:
settings = json.loads(CONFIG_FILE.read_text())
settings["prompt_file"] = str((CONFIG_FILE.parent / settings["prompt_file"]).resolve())
settings["images_dir"] = str(Path(IMAGE_DIRECTORY).resolve())
settings["output_dir"] = str(Path(OUTPUT_ROOT).resolve())
settings["backend"] = BACKEND
if MODEL is not None:
    settings["models"][BACKEND] = MODEL
print(Path(settings["prompt_file"]).read_text())
print("Mode:", MODE, "| Backend:", BACKEND, "| Model:", settings["models"][BACKEND])
print("Images:", settings["images_dir"])
print("Limit:", LIMIT if LIMIT is not None else "all")

## 4. Run

This cell streams progress. Each invocation creates a new folder; rerunning in evaluate mode makes new model calls. Stop the cell to interrupt the process. Completed records remain on disk.

In [ ]:
if MODE not in {"preview", "evaluate"}:
    raise ValueError("MODE must be preview or evaluate")
if LIMIT is not None and (type(LIMIT) is not int or LIMIT <= 0):
    raise ValueError("LIMIT must be a positive integer or None")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_folder = Path(OUTPUT_ROOT) / f"{stamp}-recognize-diagrams"
run_folder.mkdir(parents=True, exist_ok=False)
# Preserve the chosen notebook settings beside the run without modifying defaults.
run_config = run_folder / "config.json"
run_config.write_text(json.dumps(settings, indent=2) + "\n")
RUN_DIRECTORY = run_folder / "evaluation"
command = [sys.executable, str(EVAL_DIR / "run_rq1.py"),
           "--config", str(run_config), "--output-dir", str(RUN_DIRECTORY)]
if MODE == "preview":
    command.append("--dry-run")
if LIMIT is not None:
    command += ["--limit", str(LIMIT)]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
except BaseException:
    process.terminate()
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    raise
finally:
    process.stdout.close()
if return_code:
    print("Run reported errors. Inspect the saved records and diagnostics above.")
print("Saved run:", RUN_DIRECTORY)

## 5. Explore results

To inspect an older run, set `RUN_DIRECTORY` below to its `evaluation` folder. Preview runs have no model scores.

In [ ]:
# RUN_DIRECTORY = Path("/absolute/path/to/a/saved/evaluation")
manifest = json.loads((RUN_DIRECTORY / "manifest.json").read_text())
records = [json.loads(line) for line in (RUN_DIRECTORY / "results.jsonl").read_text().splitlines()]
print("Recorded:", len(records), "| Successful:", sum(r["status"] == "ok" for r in records),
      "| Errors:", sum(r["status"] == "error" for r in records))
rows = [{**record, **(record.get("result") or {})} for record in records]
show_table(rows, [("source", "Image"), ("status", "Status"), ("identification", "Identified structure"),
                  ("confidence", "Confidence"), ("visual_evidence", "Visible evidence"),
                  ("alternatives", "Alternatives"), ("error", "Error")])

## Saved files

The run folder contains the manifest, raw provider responses, individual result JSON files, and aggregate JSONL/CSV files.

The notebook is a front end to the tested scripts. Older research-question filenames remain available for compatibility; you do not need them to use this notebook.